# Notebook 05 — Controles de rigor

Este notebook no cambia ninguna decisión del proyecto: **comprueba que los resultados de los notebooks 01 a 04 no dependen de errores ni de fugas de información**. Hacemos cinco controles:

1. Que `train.csv`, `test.csv` y `test_original.csv` son exactamente las filas que la partición guardada asigna a cada conjunto en el archivo original de UCI.
2. Que no hay perfiles idénticos a la vez en entrenamiento y prueba, y que los grupos se definen solo con las entradas (nunca con la etiqueta).
3. Que ningún grupo cruza entrenamiento y validación en las diez particiones.
4. Que el pipeline no encuentra "señal" cuando no la hay: con las etiquetas mezcladas al azar debe rendir como el azar.
5. Una prueba de estrés: agrupar de forma mucho más estricta (perfiles *parecidos*, no solo idénticos) para ver cuánto del rendimiento depende de que clientes muy parecidos queden repartidos entre entrenamiento y validación.

Una aclaración sobre la prueba: este notebook **no entrena ni evalúa ningún modelo con los datos de prueba**. Solo compara los archivos de prueba con lo que se obtiene al rehacer la partición desde el original, como lo hace también el notebook 01.

In [1]:
EN_COLAB = False

if EN_COLAB:
    %pip install imbalanced-learn -q

    from google.colab import drive
    drive.mount("/content/drive")
    RUTA_BASE = "/content/drive/MyDrive/Prediccion_Abandono_Clientes/"
else:
    RUTA_BASE = "./"

RUTA_DATOS = RUTA_BASE + "data/"
RUTA_MODELOS = RUTA_BASE + "models/"
RUTA_RESULTADOS = RUTA_BASE + "resultados/"
RUTA_FIGURAS = RUTA_BASE + "figuras/"

In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from imblearn.pipeline import Pipeline

os.makedirs(RUTA_RESULTADOS, exist_ok=True)

## Cargar todo lo que dejaron los notebooks anteriores

Leemos el archivo original oficial, la partición guardada (con la asignación de cada fila y su grupo), los grupos de entrenamiento, las diez particiones de validación y la plantilla de preprocesamiento. Los nombres de las columnas originales traen espacios dobles; el diccionario `traduccion` es el mismo del notebook 01 y las pasa a los nombres en español.

In [3]:
traduccion = {
    "Call  Failure": "fallas_llamadas", "Complains": "quejas", "Subscription  Length": "antiguedad",
    "Charge  Amount": "nivel_cargo", "Seconds of Use": "segundos_uso", "Frequency of use": "frecuencia_uso",
    "Frequency of SMS": "frecuencia_sms", "Distinct Called Numbers": "numeros_distintos",
    "Age Group": "grupo_edad", "Tariff Plan": "plan_tarifario", "Age": "edad", "Churn": "abandono"
}

original = pd.read_csv(RUTA_DATOS + "original/Customer Churn.csv")
with open(RUTA_DATOS + "particion_01.json", encoding="utf-8") as archivo:
    particion = json.load(archivo)

columnas_entrada = particion["columnas_entrada"]
asignacion = np.array(particion["asignacion"])
grupos_particion = np.array(particion["grupos"])

datos = original.rename(columns=traduccion)[columnas_entrada + ["abandono"]]
filas_train = np.where(asignacion == "train")[0]
filas_test = np.where(asignacion == "test")[0]

train = pd.read_csv(RUTA_DATOS + "train.csv")
datos_grupos = pd.read_csv(RUTA_DATOS + "grupos_train.csv")
asignaciones_cv = pd.read_csv(RUTA_RESULTADOS + "02_particiones_cv.csv")
decisiones = joblib.load(RUTA_MODELOS + "decisiones_preprocesamiento.joblib")

X_train = train[columnas_entrada]
y_train = train["abandono"]
grupos_train = datos_grupos["grupo"]
print("Filas del original:", len(original), "| train:", len(filas_train), "| prueba:", len(filas_test))

Filas del original: 3150 | train: 2511 | prueba: 639


## Integridad de la partición

Si todo está bien, `train.csv` y `test.csv` son las filas del original que la partición asigna a "train" y "test", en el mismo orden y con los mismos valores. `test_original.csv` es la copia de prueba con los nombres y las columnas originales.

In [4]:
def comparar_con_original(tabla_guardada, tabla_esperada):
    """
    Recibe la tabla leída del archivo guardado y la que sale de rehacer la partición desde el original.
    Devuelve True si tienen las mismas columnas, las mismas filas y exactamente los mismos valores.
    """
    esperada = tabla_esperada.reset_index(drop=True)
    guardada = tabla_guardada.reset_index(drop=True)
    if list(guardada.columns) != list(esperada.columns):
        return False
    return bool((guardada.to_numpy() == esperada.to_numpy()).all())


test = pd.read_csv(RUTA_DATOS + "test.csv")
test_original = pd.read_csv(RUTA_DATOS + "test_original.csv")

rows = []
rows.append({"Archivo": "train.csv", "Filas": len(train),
             "Igual_al_original": comparar_con_original(train, datos.iloc[filas_train])})
rows.append({"Archivo": "test.csv", "Filas": len(test),
             "Igual_al_original": comparar_con_original(test, datos.iloc[filas_test])})
rows.append({"Archivo": "test_original.csv", "Filas": len(test_original),
             "Igual_al_original": comparar_con_original(test_original, original.iloc[filas_test])})
tabla_integridad = pd.DataFrame(rows)
tabla_integridad

,Archivo,Filas,Igual_al_original
0,train.csv,2511,True
1,test.csv,639,True
2,test_original.csv,639,True


**Lo que notamos.** Los tres archivos coinciden con el original: `train.csv` (2.511 filas), `test.csv` (639) y `test_original.csv` (639) son, valor por valor y en el mismo orden, las filas que la partición guardada asigna a entrenamiento y a prueba en el archivo oficial de UCI. La reserva de prueba es reproducible y nadie la alteró.

## Perfiles repetidos y grupos

Los clientes con exactamente las mismas entradas deben quedar juntos: si uno cae en entrenamiento y su copia en prueba, el modelo acertaría por haberlo visto. Comprobamos tres cosas: que no haya perfiles comunes entre entrenamiento y prueba, que dentro de entrenamiento cada grupo corresponda a un único perfil (los grupos se definen con las entradas, sin la etiqueta) y que ningún grupo cruce entrenamiento y validación en las diez particiones.

In [5]:
perfiles_compartidos = pd.merge(X_train.drop_duplicates(), test[columnas_entrada].drop_duplicates(), how="inner")

perfil_por_fila = X_train.astype(str).agg("|".join, axis=1)
perfiles_por_grupo = perfil_por_fila.groupby(grupos_train).nunique()
grupos_por_perfil = grupos_train.groupby(perfil_por_fila).nunique()

grupos_cruzados = 0
for numero in range(1, 11):
    en_validacion = asignaciones_cv["particion_validacion"] == numero
    comunes = set(grupos_train[en_validacion]).intersection(set(grupos_train[~en_validacion]))
    grupos_cruzados = grupos_cruzados + len(comunes)

grupos_json_train = set(grupos_particion[asignacion == "train"])
grupos_json_test = set(grupos_particion[asignacion == "test"])

tabla_grupos = pd.DataFrame([{
    "Perfiles_en_train_y_test": len(perfiles_compartidos),
    "Grupos_con_mas_de_un_perfil": int((perfiles_por_grupo > 1).sum()),
    "Perfiles_en_mas_de_un_grupo": int((grupos_por_perfil > 1).sum()),
    "Grupos_que_cruzan_particiones_de_validacion": grupos_cruzados,
    "Grupos_comunes_entre_train_y_test": len(grupos_json_train.intersection(grupos_json_test)),
}])
tabla_grupos

,Perfiles_en_train_y_test,Grupos_con_mas_de_un_perfil,Perfiles_en_mas_de_un_grupo,Grupos_que_cruzan_particiones_de_validacion,Grupos_comunes_entre_train_y_test
0,0,0,0,0,0


**Lo que notamos.** Todos los controles dan cero: no hay ningún perfil de entradas común a entrenamiento y prueba, ningún grupo tiene más de un perfil, ningún perfil se reparte en más de un grupo y ningún grupo cruza entrenamiento y validación en las diez particiones ni entre entrenamiento y prueba. Los grupos se construyeron solo con las entradas, sin la etiqueta, como exige el proyecto.

## Etiquetas mezcladas: el pipeline no debe encontrar señal donde no la hay

Mezclamos al azar la etiqueta de los clientes de entrenamiento (así deja de tener relación con sus entradas) y repetimos la validación cruzada por grupos con un Random Forest, el mismo tipo de modelo del notebook 04. Si hubiera una fuga, el ROC AUC seguiría muy por encima de 0,5. Si no, debe quedar alrededor de 0,5 y la precisión promedio alrededor de la proporción de abandonos.

In [6]:
def armar_pipeline_control(plantilla):
    """
    Arma el pipeline de control: pasos de la plantilla del notebook 02 y un Random Forest con class_weight
    balanceado. Es del mismo tipo que el modelo final, pero sin ajustar hiperparámetros: aquí no buscamos
    rendimiento, solo comprobar que no hay fuga.
    """
    pasos = list(clone(plantilla).steps)
    pasos.append(("modelo", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                  random_state=42, n_jobs=-1)))
    return Pipeline(pasos)


def probabilidades_con_particiones(plantilla, X, y, particiones):
    """
    Para cada partición entrena una copia del pipeline de control con la parte de entrenamiento y calcula la
    probabilidad de abandono de la parte de validación. Devuelve una probabilidad por fila, siempre calculada
    por un modelo que no vio esa fila.
    """
    probabilidades = np.zeros(len(y))
    for indices_train, indices_validacion in particiones:
        modelo = armar_pipeline_control(plantilla)
        modelo.fit(X.iloc[indices_train], y.iloc[indices_train])
        probabilidades[indices_validacion] = modelo.predict_proba(X.iloc[indices_validacion])[:, 1]
    return probabilidades


def resumir_metricas(y_real, probabilidades):
    """
    Devuelve el ROC AUC, la precisión promedio y el F1 con umbral 0,5 de unas probabilidades.
    Sirven para comparar los controles sin tener que elegir un umbral para cada uno.
    """
    return {"ROC_AUC": roc_auc_score(y_real, probabilidades),
            "Precision_promedio": average_precision_score(y_real, probabilidades),
            "F1_umbral_0_5": f1_score(y_real, (probabilidades >= 0.5).astype(int))}


plantilla = decisiones["preprocesamiento"]
particiones_cv = []
for numero in range(1, 11):
    particiones_cv.append((np.where(asignaciones_cv["particion_validacion"] != numero)[0],
                           np.where(asignaciones_cv["particion_validacion"] == numero)[0]))

probabilidades_reales = probabilidades_con_particiones(plantilla, X_train, y_train, particiones_cv)
y_mezclado = pd.Series(np.random.RandomState(42).permutation(y_train.to_numpy()))
probabilidades_mezcladas = probabilidades_con_particiones(plantilla, X_train, y_mezclado, particiones_cv)

fila_real = resumir_metricas(y_train, probabilidades_reales)
fila_real["Etiquetas"] = "Reales"
fila_mezclada = resumir_metricas(y_mezclado, probabilidades_mezcladas)
fila_mezclada["Etiquetas"] = "Mezcladas"
tabla_permutadas = pd.DataFrame([fila_real, fila_mezclada])
print("Proporción de abandonos:", round(y_train.mean(), 4))
tabla_permutadas.round(3)

Proporción de abandonos: 0.1561


,ROC_AUC,Precision_promedio,F1_umbral_0_5,Etiquetas
0,0.982,0.891,0.859,Reales
1,0.492,0.153,0.147,Mezcladas


**Lo que notamos.** Con las etiquetas reales el Random Forest da un ROC AUC de 0,982, una precisión promedio de 0,891 y un F1 de 0,859. Con las etiquetas mezcladas queda en un ROC AUC de 0,492, una precisión promedio de 0,153 (la proporción de abandonos es 0,156) y un F1 de 0,147: exactamente el nivel del azar. El pipeline completo no encuentra señal donde no la hay, así que su mecánica no filtra información.

## Prueba de estrés: agrupar perfiles parecidos, no solo idénticos

Agrupar por perfiles *idénticos* evita las copias exactas, pero deja que clientes casi iguales (por ejemplo, dos con los mismos datos salvo un mensaje de texto más) queden repartidos entre entrenamiento y validación. Si el rendimiento dependiera mucho de eso, estaría inflado. Para medirlo hacemos una prueba más dura: agrupamos con **rangos gruesos** (segundos de uso por miles, frecuencia de uso de a 20, mensajes de a 50, números distintos de a 10, antigüedad de a 10 meses, fallas de a 5, y las variables discretas tal cual) de modo que clientes parecidos queden siempre juntos, y repetimos la validación cruzada por esos grupos. Es más estricta de lo necesario, así que sirve como cota pesimista.

In [7]:
def crear_grupos_gruesos(X):
    """
    Agrupa a los clientes por rangos gruesos de sus entradas, de modo que los clientes parecidos (no solo los
    idénticos) queden en el mismo grupo. Devuelve un arreglo con el número de grupo de cada fila.
    """
    gruesa = pd.DataFrame({
        "segundos": X["segundos_uso"] // 1000,
        "uso": X["frecuencia_uso"] // 20,
        "sms": X["frecuencia_sms"] // 50,
        "numeros": X["numeros_distintos"] // 10,
        "quejas": X["quejas"],
        "antiguedad": X["antiguedad"] // 10,
        "grupo_edad": X["grupo_edad"],
        "plan": X["plan_tarifario"],
        "nivel_cargo": X["nivel_cargo"],
        "fallas": X["fallas_llamadas"] // 5,
    })
    return gruesa.astype(str).agg("|".join, axis=1).factorize()[0]


grupos_gruesos = crear_grupos_gruesos(X_train)
divisor = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
particiones_gruesas = []
for indices_train, indices_validacion in divisor.split(X_train, y_train, grupos_gruesos):
    particiones_gruesas.append((indices_train, indices_validacion))

probabilidades_gruesas = probabilidades_con_particiones(plantilla, X_train, y_train, particiones_gruesas)

fila_exacta = resumir_metricas(y_train, probabilidades_reales)
fila_exacta["Agrupacion"] = "Perfiles idénticos (la del proyecto)"
fila_exacta["Grupos"] = len(set(grupos_train))
fila_gruesa = resumir_metricas(y_train, probabilidades_gruesas)
fila_gruesa["Agrupacion"] = "Rangos gruesos (más estricta)"
fila_gruesa["Grupos"] = len(set(grupos_gruesos))
tabla_estres = pd.DataFrame([fila_exacta, fila_gruesa])
tabla_estres.round(3)

,ROC_AUC,Precision_promedio,F1_umbral_0_5,Agrupacion,Grupos
0,0.982,0.891,0.859,Perfiles idénticos (la del proyecto),2255
1,0.972,0.843,0.811,Rangos gruesos (más estricta),1299


**Lo que notamos.** Al agrupar por rangos gruesos quedan 1.299 grupos en vez de 2.255, y el rendimiento baja: el ROC AUC pasa de 0,982 a 0,972, la precisión promedio de 0,891 a 0,843 y el F1 con umbral 0,5 de 0,859 a 0,811. Es decir, parte del rendimiento (unos 0,05 de F1) se apoya en que clientes muy parecidos, aunque no idénticos, quedan repartidos entre entrenamiento y validación. No es una fuga de información en el sentido de un error del procedimiento, pero conviene decirlo: el F1 de 0,86 puede ser algo optimista para clientes distintos de los que ya vio el modelo. Aun en este escenario más duro el F1 es de 0,81, así que el resultado sigue siendo alto y la conclusión de que los modelos de árboles superan con claridad a los lineales no cambia.

In [8]:
tabla_integridad.to_csv(RUTA_RESULTADOS + "05_integridad_particion.csv", index=False)
tabla_grupos.to_csv(RUTA_RESULTADOS + "05_perfiles_y_grupos.csv", index=False)
tabla_permutadas.to_csv(RUTA_RESULTADOS + "05_etiquetas_permutadas.csv", index=False)
tabla_estres.to_csv(RUTA_RESULTADOS + "05_prueba_estres_grupos.csv", index=False)
print("Tablas guardadas en", RUTA_RESULTADOS)

Tablas guardadas en ./resultados/


## Conclusiones

- **Partición.** Los archivos de datos son reproducibles desde el original y coinciden valor por valor con la asignación guardada.
- **Perfiles y grupos.** No hay perfiles compartidos entre entrenamiento y prueba y ningún grupo cruza una partición.
- **Mecánica del pipeline.** Con etiquetas mezcladas, el Random Forest baja a un ROC AUC de 0,492: sin fuga.
- **Matiz que hay que decir.** Con una agrupación más estricta (perfiles parecidos juntos) el F1 baja de 0,859 a 0,811. Reportamos el resultado con la agrupación del proyecto (perfiles idénticos), que es la acordada, pero dejamos constancia de que una parte de unos 0,05 de F1 depende de la cercanía entre clientes parecidos. Una lectura prudente del F1 final es "entre 0,81 y 0,86".
- **Lo que estos controles no resuelven.** No comprueban que una entrada (como el uso igual a cero) refleje un abandono ya ocurrido, ni la generalización a otro periodo o mercado: el conjunto no trae fechas.
- **Archivos generados.** `resultados/05_integridad_particion.csv`, `resultados/05_perfiles_y_grupos.csv`, `resultados/05_etiquetas_permutadas.csv` y `resultados/05_prueba_estres_grupos.csv`.